In [20]:
import pandas as pd
import numpy as np

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

In [21]:
!tar -xzf luad_tcga_pub.tar.gz

In [22]:
!ls

luad_tcga_pub  luad_tcga_pub.tar.gz  sample_data


In [23]:
import os

for root, dirs, files in os.walk("."):
    for f in files:
        if "mutation" in f.lower():
            print(os.path.join(root, f))

./luad_tcga_pub/meta_mutations.txt
./luad_tcga_pub/data_mutations.txt


In [24]:
df = pd.read_csv(
    "./luad_tcga_pub/data_mutations.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

In [25]:
patient_gene = pd.crosstab(
    df["Tumor_Sample_Barcode"],
    df["Hugo_Symbol"]
)

patient_gene = (patient_gene > 0).astype(int)

In [26]:
gene_counts = patient_gene.sum(axis=0)

selected_genes = gene_counts[
    gene_counts >= 20
].index

patient_gene_small = patient_gene[selected_genes]

print(patient_gene_small.shape)

(230, 327)


In [27]:
patient_gene_small.sum().sort_values(
    ascending=False
).head(50)

,0
Hugo_Symbol,
TTN,129
TP53,107
MUC16,99
RYR2,95
CSMD3,90
LRP1B,80
USH2A,79
KRAS,75
ZFHX4,71


In [28]:
from scipy.stats import fisher_exact

results = []

genes = list(patient_gene_small.columns)

n_patients = len(patient_gene_small)

for i in range(len(genes)):
    for j in range(i + 1, len(genes)):

        gene_a = genes[i]
        gene_b = genes[j]

        a = patient_gene_small[gene_a]
        b = patient_gene_small[gene_b]

        both = ((a == 1) & (b == 1)).sum()
        only_a = ((a == 1) & (b == 0)).sum()
        only_b = ((a == 0) & (b == 1)).sum()
        neither = ((a == 0) & (b == 0)).sum()

        contingency = [
            [both, only_a],
            [only_b, neither]
        ]

        odds_ratio, p_value = fisher_exact(contingency)

        results.append([
            gene_a,
            gene_b,
            both,
            odds_ratio,
            p_value
        ])

results_df = pd.DataFrame(
    results,
    columns=[
        "GeneA",
        "GeneB",
        "CoOccurrence",
        "OddsRatio",
        "PValue"
    ]
)

results_df.head()

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue
0,ABCA12,ABCA13,4,1.351044,0.536418
1,ABCA12,ABCB5,2,1.116959,0.701790
2,ABCA12,ABCC11,6,5.571429,0.004708
3,ABCA12,ACAN,1,0.447619,0.701790
4,ABCA12,ACTN2,2,0.942356,1.000000


In [29]:
from statsmodels.stats.multitest import multipletests

results_df["Adj_P"] = multipletests(
    results_df["PValue"],
    method="fdr_bh"
)[1]

results_df.head()

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
0,ABCA12,ABCA13,4,1.351044,0.536418,0.659401
1,ABCA12,ABCB5,2,1.116959,0.701790,0.824567
2,ABCA12,ABCC11,6,5.571429,0.004708,0.032761
3,ABCA12,ACAN,1,0.447619,0.701790,0.824567
4,ABCA12,ACTN2,2,0.942356,1.000000,1.000000


In [30]:
sig_pairs = results_df[
    results_df["Adj_P"] < 0.05
]

print("Significant pairs:", len(sig_pairs))

sig_pairs.sort_values(
    "OddsRatio",
    ascending=False
).head(30)

Significant pairs: 10374


,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
45034,NDST4,ZFHX4,20,inf,7.478565e-12,3.986150e-07
35275,HGF,SEMA5A,13,28.600000,6.655439e-10,7.094831e-06
17799,COL3A1,CSMD3,26,28.031250,3.677919e-10,4.900919e-06
51131,SLC39A12,TTN,24,22.857143,6.171427e-06,1.289974e-03
38398,LAMA2,TTN,24,22.857143,6.171427e-06,1.289974e-03
36438,ITGAX,TSHZ2,11,18.521368,1.342539e-07,2.168444e-04
21901,DCHS1,TTN,20,18.348624,9.793845e-05,4.291490e-03
9902,ATRNL1,TP53,24,17.493976,2.951641e-07,3.691094e-04
42101,MGAM,TTN,19,17.272727,1.019366e-04,4.338444e-03
41495,MAP2,TTN,19,17.272727,1.019366e-04,4.338444e-03


In [31]:
important_genes = [
    "TP53",
    "KRAS",
    "EGFR",
    "STK11",
    "KEAP1",
    "MET",
    "BRAF",
    "PIK3CA",
    "RB1",
    "ALK",
    "ROS1",
    "RET",
    "ERBB2",
    "SMARCA4",
    "NF1"
]
sig_pairs[
    (
        sig_pairs["GeneA"].isin(important_genes)
    ) &
    (
        sig_pairs["GeneB"].isin(important_genes)
    )
].sort_values(
    "Adj_P"
)

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
25864,EGFR,KRAS,2,0.101287,0.000123,0.004675
37949,KRAS,NF1,2,0.124266,0.000649,0.011287
25989,EGFR,STK11,0,0.000000,0.001123,0.015099
38025,KRAS,STK11,21,2.783626,0.004966,0.033966
45392,NF1,TP53,21,3.093023,0.006245,0.038162
38042,KRAS,TP53,25,0.445122,0.007194,0.041900
5989,ALK,TP53,17,3.683333,0.007378,0.042672
52042,STK11,TP53,11,0.371408,0.008858,0.047063


In [33]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [34]:
sig_pairs.to_csv(
    "/content/drive/MyDrive/Cancer Evolution Arena/Data/significant_pairs.csv",
    index=False
)